# ⚡ Microgrid Monitoring — Nostradamus IoTO SDK

Solar/wind/battery microgrid: two solar arrays, a wind turbine and a battery bank report power output, cumulative energy, voltage, current, temperature, efficiency, grid frequency and status.

This notebook mirrors `examples/smart_energy/microgrid_monitoring.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **4 assets × 48 half-hourly readings** (192 records) to the collection
`microgrid_assets`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `microgrid_assets` collection with a schema
2. Generate and send 4 assets × 48 half-hourly readings
3. Query solar output, low battery SOC, faults and peak production
4. Aggregate power output, temperature and battery SOC statistics
5. Delete a 12-hour window for `SOLAR_ARRAY_01`
6. Delete the collection

## What this notebook highlights

- Realistic day/night solar curves and wind gusts
- Battery charge/discharge cycles with SOC tracking
- Fault detection via the `status` flag

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import math
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("⚡ Nostradamus IoTO SDK — Microgrid Monitoring")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the asset collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/smart_energy/microgrid_monitoring.py`.

In [ ]:
COLLECTION_NAME = "microgrid_assets"

COLLECTION_SCHEMA = {
                    "key": "SOLAR_ARRAY_01",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "power_output_kw": 35.5,
                    "energy_cumulative_kwh": 150.0,
                    "voltage_v": 480.0,
                    "current_a": 73.9,
                    "temperature_c": 45.0,
                    "efficiency_pct": 19.5,
                    "solar_irradiance_wm2": 850.0,
                    "wind_speed_ms": 0.0,
                    "battery_soc_pct": 0.0,
                    "grid_frequency_hz": 50.01,
                    "status": 1,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Solar/wind/battery microgrid data",
            tags=["energy", "microgrid", "renewable"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send solar/wind/battery data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_microgrid_data()` is the
same generator used by the script.

In [ ]:
def generate_microgrid_data(num_readings=48):
    """Generate microgrid asset data with realistic energy patterns.

    Args:
        num_readings: Readings per asset (30-min intervals)

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)
    assets = [
        "SOLAR_ARRAY_01",
        "SOLAR_ARRAY_02",
        "WIND_TURBINE_01",
        "BATTERY_BANK_01",
    ]

    for asset_id in assets:
        is_solar = asset_id.startswith("SOLAR")
        is_wind = asset_id.startswith("WIND")
        is_battery = asset_id.startswith("BATTERY")
        cumulative_kwh = 0.0
        battery_soc = 70.0

        for i in range(num_readings):
            timestamp = base_time + timedelta(minutes=30 * i)
            hour = timestamp.hour + timestamp.minute / 60.0

            has_fault = random.random() < 0.02

            if is_solar:
                solar_angle = (
                    max(0, math.sin(math.pi * (hour - 6) / 12))
                    if 6 <= hour <= 18
                    else 0.0
                )
                irradiance = solar_angle * 900 + random.uniform(-80, 80)
                irradiance = max(0, min(1000, irradiance))
                power = irradiance / 1000 * 50 * random.uniform(0.85, 1.0)
                power = round(max(0, min(50, power)), 2)
                voltage = 350 + power * 5 + random.uniform(-10, 10)
                voltage = round(min(600, max(300, voltage)), 1)
                temp = 20 + power * 1.0 + random.uniform(-3, 5)
                efficiency = round(random.uniform(15, 22), 1) if power > 0 else 0.0
            elif is_wind:
                irradiance = 0
                wind_speed = abs(random.gauss(8, 4) + random.uniform(-2, 2))
                wind_speed = round(min(25, max(0, wind_speed)), 1)
                if wind_speed < 3:
                    power = 0
                elif wind_speed > 20:
                    power = 0
                else:
                    power = (wind_speed / 15) ** 3 * 100
                    power = round(
                        min(100, max(0, power)) * random.uniform(0.8, 1.0),
                        2,
                    )
                voltage = round(380 + random.uniform(-5, 40), 1)
                temp = 25 + power * 0.3 + random.uniform(-5, 10)
                efficiency = round(random.uniform(25, 45), 1) if power > 0 else 0.0
            else:
                irradiance = 0
                wind_speed = 0
                solar_angle = (
                    max(0, math.sin(math.pi * (hour - 6) / 12))
                    if 6 <= hour <= 18
                    else 0.0
                )
                if solar_angle > 0.3:
                    power = round(random.uniform(5, 30), 2)
                    battery_soc = min(100, battery_soc + power * 0.3)
                else:
                    power = round(-random.uniform(5, 25), 2)
                    battery_soc = max(20, battery_soc + power * 0.3)
                voltage = round(
                    48 + (battery_soc - 50) * 0.16 + random.uniform(-0.5, 0.5),
                    1,
                )
                temp = 25 + abs(power) * 0.5 + random.uniform(-3, 5)
                efficiency = round(random.uniform(85, 98), 1)

            current = round(abs(power) * 1000 / max(voltage, 1), 1)
            cumulative_kwh += abs(power) * 0.5

            reading = {
                "key": asset_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "power_output_kw": power,
                "energy_cumulative_kwh": round(cumulative_kwh, 1),
                "voltage_v": voltage,
                "current_a": current,
                "temperature_c": round(min(75, max(15, temp)), 1),
                "efficiency_pct": efficiency,
                "grid_frequency_hz": round(50.0 + random.uniform(-0.2, 0.2), 2),
                "status": 0 if has_fault else 1,
            }

            if is_solar:
                reading["solar_irradiance_wm2"] = round(irradiance, 0)
            if is_wind:
                reading["wind_speed_ms"] = wind_speed
            if is_battery:
                reading["battery_soc_pct"] = round(battery_soc, 1)

            data.append(reading)

    return data

data = generate_microgrid_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **☀️ SOLAR_ARRAY_01 readings**
- **🔋 Low battery SOC (< 30 %)**
- **🚨 Faults (status == 0)**
- **🌞 Peak production (irradiance > 700 W/m² and power > 35 kW)**
- **Top 5 power output readings**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # ☀️ SOLAR_ARRAY_01 readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "SOLAR_ARRAY_01"},
        ],
    )
    summary(result, "☀️ SOLAR_ARRAY_01 readings")

    # 🔋 Low battery SOC (< 30 %)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "battery_soc_pct", "operator": "lt", "property_value": 30},
        ],
    )
    summary(result, "🔋 Low battery SOC (< 30 %)")

    # 🚨 Faults (status == 0)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "status", "operator": "eq", "property_value": 0},
        ],
    )
    summary(result, "🚨 Faults (status == 0)")

    # 🌞 Peak production (irradiance > 700 W/m² and power > 35 kW)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "solar_irradiance_wm2", "operator": "gt", "property_value": 700},
            {"property_name": "power_output_kw", "operator": "gt", "property_value": 35},
        ],
    )
    summary(result, "🌞 Peak production (irradiance > 700 W/m² and power > 35 kW)")

    # Top 5 power output readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "power_output_kw"],
        order_by='{"field": "power_output_kw", "order": "desc"}',
        limit=5,
    )
    print("🏆 Top 5 power output readings")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average power output (kW)", "power_output_kw", StatOperation.AVG),
        ("Maximum temperature (°C)", "temperature_c", StatOperation.MAX),
        ("Minimum battery SOC (%)", "battery_soc_pct", StatOperation.MIN),
        ("Distinct assets", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one asset

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`SOLAR_ARRAY_01` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "SOLAR_ARRAY_01"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")